In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np

base_dir = "/content/drive/MyDrive/Dataset Protein"

cb513 = np.load(base_dir + "/cb513+profile_split1.npy.gz", allow_pickle=True)
cullpdb = np.load(base_dir + "/cullpdb+profile_5926_filtered.npy.gz", allow_pickle=True)

print(cb513.shape)
print(cullpdb.shape)

/tmp/ipykernel_7335/3578813769.py:5: UserWarning: Reading `.npy` or `.npz` file required additional header parsing as it was created on Python 2. Save the file again to speed up loading and avoid this warning.
  cb513 = np.load(base_dir + "/cb513+profile_split1.npy.gz", allow_pickle=True)


(514, 39900)
(5365, 39900)


In [ ]:
#Reshape
dataset_test = cb513.reshape(cb513.shape[0], 700, 57)
dataset_train = cullpdb.reshape(cullpdb.shape[0], 700, 57)

print(dataset_test.shape)
print(dataset_train.shape)

(514, 700, 57)
(5365, 700, 57)


In [ ]:
#Handle Padding
X_test = dataset_test[:, :, :21]
X_test = np.concatenate([X_test, dataset_test[:, :, 24:]], axis=2)

y_test = dataset_test[:, :, 21:24]

X_train = dataset_train[:, :, :21]   # amino acid
X_train = np.concatenate([X_train, dataset_train[:, :, 24:]], axis=2)

y_train = dataset_train[:, :, 21:24]

mask_train = np.sum(X_train, axis=2) != 0
mask_test  = np.sum(X_test, axis=2) != 0
sample_weight = mask_train.astype(float)

In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (5365, 700, 54)
y_train: (5365, 700, 3)


In [ ]:
print(np.sum(dataset_train[0][0][42:50]))

0.360779776237905


In [ ]:
print(dataset_train.shape)
print(dataset_train[0][0])

(5365, 700, 57)
[0.         0.         0.         0.         0.         1.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         1.         0.
 0.         0.         0.         0.         0.         0.
 0.         1.         0.         1.         1.         0.02145729
 0.01365366 0.0054863  0.01212843 0.99848372 0.0066267  0.2748805
 0.0765622  0.01448573 0.11815697 0.09708863 0.01261716 0.00570892
 0.01852383 0.01763634 0.02104135 0.02484354 0.05625294 0.51499552
 0.26894143 0.99938297 0.        ]


In [ ]:
print("21:24 =", dataset_train[0][0][21:24])
print("42:50 =", dataset_train[0][0][42:50])

21:24 = [0. 1. 0.]
42:50 = [0.0765622  0.01448573 0.11815697 0.09708863 0.01261716 0.00570892
 0.01852383 0.01763634]


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, Dropout, TimeDistributed, Dense

input_layer = Input(shape=(700, 54))

x = Conv1D(64, 3, padding='same', activation='relu')(input_layer)
x = Conv1D(128, 3, padding='same', activation='relu')(x)

x = Dropout(0.3)(x)

x = TimeDistributed(Dense(64, activation='relu'))(x)
output = TimeDistributed(Dense(3, activation='softmax'))(x)

model = Model(inputs=input_layer, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
def data_generator(data, batch_size=16):
    num_samples = data.shape[0]

    while True:
        for i in range(0, num_samples, batch_size):
            batch = data[i:i+batch_size]

            batch = batch.reshape(batch.shape[0], 700, 57)

            X = np.concatenate([batch[:, :, :21], batch[:, :, 24:]], axis=2)

            y = batch[:, :, 21:24]

            mask = np.sum(X, axis=2) != 0
            sample_weight = mask.astype(float)

            yield X.astype(np.float32), y.astype(np.float32), sample_weight

In [ ]:
model.fit(
    X_train, y_train,
    sample_weight=sample_weight,
    validation_data=(X_test, y_test, mask_test.astype(float)),
    epochs=20,
    batch_size=16
)

Epoch 1/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1082s 2s/step - accuracy: 0.7490 - loss: 0.0314 - val_accuracy: 0.8142 - val_loss: 0.0123
Epoch 2/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 30s 62ms/step - accuracy: 0.7515 - loss: 0.0106 - val_accuracy: 0.8142 - val_loss: 0.0132
Epoch 3/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 21s 61ms/step - accuracy: 0.7515 - loss: 0.0104 - val_accuracy: 0.8142 - val_loss: 0.0136
Epoch 4/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - accuracy: 0.7515 - loss: 0.0102 - val_accuracy: 0.8142 - val_loss: 0.0124
Epoch 5/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 20s 60ms/step - accuracy: 0.7515 - loss: 0.0100 - val_accuracy: 0.8141 - val_loss: 0.0129
Epoch 6/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 20s 61ms/step - accuracy: 0.7516 - loss: 0.0098 - val_accuracy: 0.8140 - val_loss: 0.0134
Epoch 7/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 20s 59ms/step - accuracy: 0.7516 - loss: 0.0097 - val_accuracy: 0.8142 - val_loss: 0.0122
Epoch 8/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 20s 61ms/step - accuracy: 0.7516 - loss: 0.0095 - 

In [ ]:
model.evaluate(X_test, y_test)

17/17 ━━━━━━━━━━━━━━━━━━━━ 158s 5s/step - accuracy: 0.8135 - loss: 0.0131


[0.013061665929853916, 0.8134880065917969]